In [1]:
import mappy as mp
import random

# --- Your data generation (unchanged) ---
def create_test_data():
    bases = ['A', 'C', 'G', 'T']
    target_seq = "".join(random.choice(bases) for _ in range(500))

    q1 = "ACGACTAGCACGATAGCACTACGACAGC"
    q2 = "GACTTTCGACGACCTCTCTAGCAAAGCC"

    target_list = list(target_seq)
    target_list[100:100+len(q1)] = list(q1)
    target_list[350:350+len(q2)] = list(q2)

    # Returning distinct queries so we can distinguish 5' from 3'
    return "".join(target_list), [q1, q2]

ref_seq, flanks = create_test_data()
f5 = flanks[0]
f3 = flanks[1]

In [43]:
import parasail
import mappy as mp
import sys

# Define a simple namedtuple to mimic mappy's hit object for compatibility
from collections import namedtuple
Hit = namedtuple('Hit', ['r_st', 'r_en', 'strand', 'score'])

def get_best_hit_parasail(query_seq, target_seq):
    """
    Replaces mappy's get_best_hit using Smith-Waterman.
    Checks both strands and returns the best alignment.
    """
    best_score = -1
    best_hit = None
    
    # Define scoring (Match=2, Mismatch=-1, Open=4, Extend=2)
    # These are standard for noisy Nanopore alignment
    matrix = parasail.dnafull
    gap_open = 4
    gap_extend = 2

    # Check Forward and Reverse Complement
    strands = [(query_seq, 1), (mp.revcomp(query_seq), -1)]

    for seq, strand_val in strands:
        # Use local alignment (Smith-Waterman)
        result = parasail.sw_trace_striped_16(seq, target_seq, gap_open, gap_extend, matrix)
        
        # Parasail returns the end position of the alignment in the target
        # We can extract the start position from the traceback
        if result.score > best_score:
            # Simple heuristic: Score should be at least 2x length of flank
            # You can adjust this threshold based on noise levels
            if result.score > (len(query_seq) * 1.2): 
                best_score = result.score
                
                # Traceback gives us the target start and end
                # Note: parasail indices are 0-based
                best_hit = Hit(
                    r_st=result.end_ref - len(seq), # Approximation if traceback is messy
                    r_en=result.end_ref,
                    strand=strand_val,
                    score=result.score
                )
    # print(best_hit)
    return best_hit

def extract_region(read_seq, flank_5p, flank_3p):
    """
    Extracts sequence between two flanking sequences using Parasail.
    """
    h5 = get_best_hit_parasail(flank_5p, read_seq)
    h3 = get_best_hit_parasail(flank_3p, read_seq)

    if not h5 or not h3:
        return None, "Flanks not found"

    if h5.strand != h3.strand:
        return None, "Flank orientation mismatch"

    # Forward Strand
    if h5.strand == 1:
        if h5.r_en >= h3.r_st:
            return None, "Negative distance"
        return read_seq[h5.r_en : h3.r_st], "+"

    # Reverse Strand
    else:
        if h3.r_en >= h5.r_st:
            return None, "Negative distance (RC)"
        rc_segment = read_seq[h3.r_en : h5.r_st]
        return mp.revcomp(rc_segment), "-"

def process_fastq(fastq_path, bc_5p, bc_3p, ins_5p, ins_3p, min_length_fastq=3000):
    print(f"Processing: {fastq_path}")
    print("Read_ID\tBC_Status\tBC_Seq\tIns_Status\tIns_Seq")

    pairs = []

    i = 0
    for name, seq, qual in mp.fastx_read(fastq_path):
        if len(seq) < min_length_fastq:
            continue

        # print("\n\n")
        # print(seq)
        
        i += 1
        
        # Extract Barcode
        bc_seq, bc_status = extract_region(seq, bc_5p, bc_3p)
        
        # Extract Insert
        ins_seq, ins_status = extract_region(seq, ins_5p, ins_3p)

        bc_out = bc_seq if bc_seq else "NA"
        ins_out = ins_seq if ins_seq else "NA"

        pairs.append([bc_out, ins_out])

        # print(f"{name[:10]}...\t{bc_status}\t{bc_out[:20]}...\t{ins_status}\t{ins_out[:20]}...")

        # if i >= 10: 
        #     break

        if i % 1000 == 0:
            print(i)

    return pairs

if __name__ == "__main__":
    # Your flank sequences
    barcode_5p = "CAGCTGACGAGTCCCAAATAGGACGAgACGCGC".upper()
    barcode_3p = "cGTAAACTGGATCCGCAGGCCTCTGCTAGCTTGACTG".upper()
    insert_5p  = "CTTGGTGCCAGCTTATCA".upper()
    insert_3p  = "cctatgaagtgctctagtcaagtttaact".upper()

    fastq_file = "/Users/ogw/Downloads/ris_plasmids/no_sample_id/20251208_1553_MN41644_AYO707_6ec906fc/fastq_pass/combined.fastq.gz"  # Update with your path
    pairs = process_fastq(fastq_file, barcode_5p, barcode_3p, insert_5p, insert_3p)



Processing: /Users/ogw/Downloads/ris_plasmids/no_sample_id/20251208_1553_MN41644_AYO707_6ec906fc/fastq_pass/combined.fastq.gz
Read_ID	BC_Status	BC_Seq	Ins_Status	Ins_Seq
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000


In [ ]:
for pair in pairs:
    bc = pair[0]
    insert = pair[1]
    if bc is None or insert is None:
        continue

    print("")
    print(len(bc))
    print(len(insert))

In [36]:
matrix = parasail.dnafull
result = parasail.sw_trace_striped_16("ACGACTGACACTGACACGACATCGACACTTCTCTCGCGGCGCG", "ACGACTGACACTGCTTCTCTCGCGGCGCG", 4, 0, matrix)
print(f"Score with Ns: {result.score}")

Score with Ns: 141
